# 03b 相关性高级输出（分区 + 上下三角拼图，复刻原 notebook 风格）

本 notebook 做两类输出：
1) **分区相关性热图**：按 `crop × ph_bin` 逐个分区输出热图；
2) **上下三角拼图对比**（更像原版）：
   - 同一 pH 分区内：下三角=玉米，上三角=马铃薯（若两者都存在）；

输出目录：`outputs/corr_advanced/`，并可自动打包成 `corr_advanced_all.pdf`（可在 config.yaml 调参）。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')


In [2]:
# 读取清洗后的数据（优先 bcf_final.parquet；没有就用 clean_data.parquet）
p1 = OUT/'bcf_final.parquet'
p2 = OUT/'clean_data.parquet'
df = pd.read_parquet(p1) if p1.exists() else pd.read_parquet(p2)
df.shape


(5045, 29)

In [3]:
from src.corr_advanced import build_group_correlations, export_triangle_comparisons, export_corr_pdf

# 你可以在这里指定“优先展示的变量”（若存在就前排），更像论文写法
prefer_cols = []
corr_maps = build_group_correlations(df, cfg, OUT, group_keys=('crop','ph_bin'), prefer_cols=prefer_cols)

len(corr_maps)


8

In [4]:
# 生成“同 pH 分区：玉米 vs 马铃薯”的上下三角拼图（如果两作物都有）
out_dir = (OUT / (cfg.get('corr_advanced', {}).get('out_subdir','corr_advanced')))
pairs = []
for ph in ['strong_acid','acid','neutral','alkaline']:
    a = ('corn', ph)
    b = ('potato', ph)
    pairs.append((a,b))

tri_pngs = export_triangle_comparisons(
    corr_maps=corr_maps,
    out_dir=out_dir,
    title_prefix='Triangle correlation compare (original-style)',
    pairs=pairs,
    out_prefix='corn_vs_potato_same_ph'
)
len(tri_pngs)


4

In [5]:
# 打包成一个总 PDF（可选，默认开启）
pdf_path = export_corr_pdf(OUT, cfg)
pdf_path


WindowsPath('../outputs/corr_advanced/corr_advanced_all.pdf')

✅ 你现在应该在 `outputs/corr_advanced/` 看到：
- `corr_*.png`：每个分区一张热图
- `triangle_*.png`：上下三角拼图对比（若两作物齐全）
- `corr_advanced_all.pdf`：一键汇总

如果你想让变量顺序更像原版（更“块状”），把 config.yaml 里的 `corr_advanced.cluster: true` 保持开启即可。